# EcoFarm 테마파크 응급상황 AI 대응 모델 — 전처리

| 항목 | 내용 |
|------|------|
| 원시 데이터 | `ecofarm_merged_dataset.csv` |
| 출력 파일 | `ecofarm_preprocessed_master.csv` |
| 목적 | 4단계 예측 파이프라인 공통 전처리 |

### 파이프라인 구성
```
[1단계] 응급 발생 여부 예측   → emergency_occurred (전체 21,900건)
[2단계] 응급 유형 예측        → emergency_type_enc (응급 발생 836건만)
[3단계] 중증도 예측           → severity_enc       (응급 발생 836건만)
[4단계] 대응 방안 제시        → 규칙 기반 + 예측 결과 결합
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print('라이브러리 로드 완료')

---
## 1. 원시 데이터 로드

In [ ]:
df = pd.read_csv('ecofarm_merged_dataset.csv')

print(f'Shape: {df.shape}')
print(f'기간: {df["date"].min()} ~ {df["date"].max()}')
print(f'시설 수: {df["facility_id"].nunique()}개')
df.head(3)

---
## 2. 결측값 처리

- `emergency_types`, `max_severity`: 응급 미발생 레코드(96.18%)는 NaN → `'none'`으로 채움
- `avg_response_time_min`: 응급 발생 후에야 알 수 있는 사후 정보 → NaN 유지 (1·2·3단계 모델 피처에서 제외)

In [ ]:
print('=== 처리 전 결측값 ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

df['emergency_types'] = df['emergency_types'].fillna('none')
df['max_severity']    = df['max_severity'].fillna('none')

print('\n=== 처리 후 ===')
remaining = df.isnull().sum()[df.isnull().sum() > 0]
print(remaining)
print('→ avg_response_time_min만 결측 (응급 미발생 레코드 — 정상)')

---
## 3. 타입 변환

In [ ]:
df['date'] = pd.to_datetime(df['date'])

for col in ['is_rain', 'is_heatwave', 'is_coldwave']:
    df[col] = df[col].astype(int)

print('date → datetime64 변환 완료')
print('bool(is_rain, is_heatwave, is_coldwave) → int 변환 완료')

---
## 4. 날짜 파생 피처

In [ ]:
df['month']      = df['date'].dt.month
df['dayofweek']  = df['date'].dt.dayofweek   # 0=월요일, 6=일요일
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

print('추가된 피처: month, dayofweek, is_weekend')
df[['date', 'month', 'dayofweek', 'is_weekend']].head(3)

---
## 5. 범주형 인코딩

| 컬럼 | 방법 | 근거 |
|------|------|------|
| `day_type` | 이진 (평일=0, 휴일=1) | 카디널리티 2 |
| `season` | 순서형 (겨울=1 ~ 여름=4) | EDA상 여름 발생률이 뚜렷하게 높음 |
| `facility_type` | 원-핫 | 시설마다 응급 패턴이 다르며 순서 없음 |

In [ ]:
df['day_type_enc'] = df['day_type'].map({'평일': 0, '휴일': 1})
df['season_enc']   = df['season'].map({'겨울': 1, '봄': 2, '가을': 3, '여름': 4})

df = pd.get_dummies(df, columns=['facility_type'], prefix='fac', dtype=int)

fac_cols = [c for c in df.columns if c.startswith('fac_')]
print(f'인코딩 완료: day_type_enc, season_enc, {fac_cols}')

---
## 6. 타깃 변수 인코딩 (2·3단계 모델용)

In [ ]:
# [2단계] 응급 유형 인코딩 — LabelEncoder
le_type = LabelEncoder()
df['emergency_type_enc'] = le_type.fit_transform(df['emergency_types'])

type_map = dict(zip(le_type.classes_, le_type.transform(le_type.classes_)))
print('[응급 유형 인코딩]')
for k, v in type_map.items():
    count = (df['emergency_types'] == k).sum()
    print(f'  {v}: {k} ({count}건)')

# [3단계] 중증도 인코딩 — 순서형
severity_map = {'none': 0, '경증': 1, '중등증': 2, '중증': 3}
df['severity_enc'] = df['max_severity'].map(severity_map)

print('\n[중증도 인코딩]')
print(df['severity_enc'].value_counts().sort_index())

---
## 7. 피처 엔지니어링

EDA에서 확인된 주요 위험 요인 조합을 피처로 추가합니다.

In [ ]:
# 기상 × 밀도 상호작용
df['heat_density'] = df['is_heatwave'] * df['density']   # 폭염 시 밀집도 (발생률 9.22%)
df['rain_density'] = df['is_rain']      * df['density']   # 강우 시 밀집도
# cold_density 제외 — 데이터 기간 내 한파 발생 0건으로 상수 피처

# 시간/날짜 위험 조합
df['holiday_summer'] = ((df['day_type_enc'] == 1) & (df['season'] == '여름')).astype(int)
df['is_peak_hour']   = df['hour'].isin([13, 15]).astype(int)   # 13시(5.16%), 15시(5.25%)

# 밀도 기반
df['is_high_density']    = (df['density'] > 0.6).astype(int)   # 0.6 초과 시 발생률 급등
df['density_risk_ratio'] = (df['density'] / df['base_risk']).round(4)
df['weighted_risk']      = (df['density'] * df['base_risk']).round(6)

# 공간 피처
# 입구(주차장/매표소 F06: x=5, y=5)까지 거리 — 대응 인력 이동 시간과 상관
df['dist_from_gate'] = np.sqrt((df['x'] - 5)**2 + (df['y'] - 5)**2).round(3)

# 파크 중심까지 거리 (전체 시설 좌표 평균)
cx = df.groupby('facility_id')['x'].first().mean()
cy = df.groupby('facility_id')['y'].first().mean()
df['dist_from_center'] = np.sqrt((df['x'] - cx)**2 + (df['y'] - cy)**2).round(3)

new_feats = [
    'heat_density', 'rain_density', 'holiday_summer',
    'is_peak_hour', 'is_high_density', 'density_risk_ratio', 'weighted_risk',
    'dist_from_gate', 'dist_from_center'
]
print(f'피처 엔지니어링 완료 ({len(new_feats)}개):')
df[new_feats].describe().round(3).T[['min', 'mean', 'max']]

---
## 8. 누수 검증 및 컬럼 역할 정리

In [ ]:
FEATURE_COLS = [
    # 시간
    'hour', 'month', 'dayofweek', 'is_peak_hour',
    # 날씨
    'is_rain', 'temperature', 'is_heatwave',
    # 시설
    'capacity', 'x', 'y', 'base_risk',
    'fac_camping', 'fac_experience', 'fac_gate', 'fac_observatory', 'fac_playground',
    # 방문객
    'visitor_count', 'density',
    # 인코딩
    'day_type_enc', 'season_enc',
    # 엔지니어링
    'heat_density', 'rain_density',
    'holiday_summer', 'is_high_density',
    'density_risk_ratio', 'weighted_risk',
    'dist_from_gate', 'dist_from_center',
    # 제외: is_weekend (day_type_enc와 상관 0.93 중복)
    # 제외: is_coldwave, cold_density (기간 내 한파 0건 — 상수)
]

META_COLS    = ['date', 'facility_id', 'facility_name', 'day_type', 'season']
LEAKAGE_COLS = ['emergency_count', 'emergency_types', 'max_severity', 'avg_response_time_min']
TARGET_COLS  = ['emergency_occurred', 'emergency_type_enc', 'severity_enc']

summary = pd.DataFrame([
    {'컬럼 그룹': '공통 피처',   '컬럼 수': len(FEATURE_COLS), '비고': '1·2·3단계 모두 사용'},
    {'컬럼 그룹': '메타데이터', '컬럼 수': len(META_COLS),    '비고': '시계열 분할용, 모델 입력 제외'},
    {'컬럼 그룹': '타깃 변수',  '컬럼 수': len(TARGET_COLS),  '비고': 'emergency_occurred(1단계), emergency_type_enc(2단계), severity_enc(3단계)'},
    {'컬럼 그룹': '누수 제외',  '컬럼 수': len(LEAKAGE_COLS), '비고': '사고 발생 후에야 알 수 있는 정보 — CSV에서도 제거됨'},
])
print(summary.to_string(index=False))

---
## 9. 시계열 분할 검증

날짜 기반 분할 — 랜덤 분할 시 미래 정보 누수 발생하므로 반드시 cutoff 방식 사용

In [ ]:
train = df[df['date'] < '2026-05-01']
val   = df[(df['date'] >= '2026-05-01') & (df['date'] < '2026-07-01')]
test  = df[df['date'] >= '2026-07-01']

split_summary = pd.DataFrame([
    {
        '분할': name,
        '기간': f"{subset['date'].min().date()} ~ {subset['date'].max().date()}",
        '전체 건수': len(subset),
        '응급 발생': int(subset['emergency_occurred'].sum()),
        '응급 발생률(%)': round(subset['emergency_occurred'].mean() * 100, 2)
    }
    for name, subset in [('Train', train), ('Val', val), ('Test', test)]
])
print(split_summary.to_string(index=False))

---
## 10. 저장

In [ ]:
# 불필요 컬럼 제거
# 누수 컬럼 — 사고 발생 후에야 알 수 있는 정보
df = df.drop(columns=['emergency_count', 'emergency_types', 'max_severity'])
# 상수 피처 — 기간 내 한파 0건
df = df.drop(columns=['is_coldwave', 'cold_density'], errors='ignore')
# 중복 피처 — day_type_enc와 상관 0.93
df = df.drop(columns=['is_weekend'])

df.to_csv('ecofarm_preprocessed_master.csv', index=False, encoding='utf-8-sig')

print('✅ 저장 완료: ecofarm_preprocessed_master.csv')
print(f'   Shape: {df.shape}')
print(f'   결측값: {df.drop(columns=["avg_response_time_min"]).isnull().sum().sum()}개 (avg_response_time_min 제외 시)')
print()
print('=== 단계별 사용 가이드 ===')
print('[1단계] 응급 발생 예측')
print('  X = df[FEATURE_COLS]')
print('  y = df["emergency_occurred"]')
print()
print('[2단계] 응급 유형 예측  (emergency_occurred == 1인 836건만)')
print('  emerg = df[df["emergency_occurred"] == 1]')
print('  X = emerg[FEATURE_COLS]')
print('  y = emerg["emergency_type_enc"]')
print()
print('[3단계] 중증도 예측  (emergency_occurred == 1인 836건만)')
print('  emerg = df[df["emergency_occurred"] == 1]')
print('  X = emerg[FEATURE_COLS]')
print('  y = emerg["severity_enc"]')